# 앙상블 검색기(Ensemble Retriever)

`EnsembleRetriever`는 여러 검색기를 결합하여 더 강력한 검색 결과를 제공하는 LangChain의 기능입니다. 이 검색기는 다양한 검색 알고리즘의 장점을 활용하여 단일 알고리즘보다 더 나은 성능을 달성할 수 있습니다.

**주요 특징**
1. 여러 검색기 통합: 다양한 유형의 검색기를 입력으로 받아 결과를 결합합니다.
2. 결과 재순위화: [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) 알고리즘을 사용하여 결과의 순위를 조정합니다.
3. 하이브리드 검색: 주로 `sparse retriever`(예: BM25)와 `dense retriever`(예: 임베딩 유사도)를 결합하여 사용합니다.

**장점**
- Sparse retriever: 키워드 기반 검색에 효과적
- Dense retriever: 의미적 유사성 기반 검색에 효과적

이러한 상호 보완적인 특성으로 인해 `EnsembleRetriever`는 다양한 검색 시나리오에서 향상된 성능을 제공할 수 있습니다.

자세한 내용은 [LangChain 공식 문서](https://python.langchain.com/docs/modules/data_connection/retrievers/ensemble)를 참조하세요.


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

- `EnsembleRetriever`를 초기화하여 `BM25Retriever`와 `FAISS` 검색기를 결합합니다. 각 검색기의 가중치를 설정됩니다.

In [2]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple's macbook",
]

bm25_retriever = BM25Retriever.from_texts(
    doc_list,
)
#  BM25Retriever의 검색 결과 개수를 1로 설정
bm25_retriever.k = 1

embedding = OpenAIEmbeddings()
faiss_vectorstore = FAISS.from_texts(
    doc_list,
    embedding,
)

faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k":1})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7, 0.3],
)

In [3]:
query = "my favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apples

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apples



In [4]:
query = "Apple company makes my favorite iphone"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

print("[Ensemble Retriever]")
for doc in ensemble_result:
    print(f"Content: {doc.page_content}")
    print()

print(["BM25 Retriever"])
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print(["FAISS Retriever"])
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apple's iphone

['BM25 Retriever']
Content: Apple is my favorite company

['FAISS Retriever']
Content: I like apple's iphone



## 런타임 Config 변경

런타임에서도 retriever 의 속성을 변경할 수 있습니다. 이는 `ConfigurableField` 클래스를 사용하여 가능합니다.

- `weights` 매개변수를 `ConfigurableField` 객체로 정의합니다.
  - 필드의 ID는 "ensemble_weights"로 설정합니다.


In [ ]:
from langchain_core.runnables import ConfigurableField

ensemble_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,    # 0번째
        faiss_retriever],  # 1번째
).configurable_fields(
    weights=ConfigurableField(
        id="ensemble_weights",
        name="Ensamble Weights",
        description="Ensamble Weights",
    )
)

In [15]:
bm25_docs = bm25_retriever.invoke(
    "my favorite fruit is apple"
)

faiss_docs = faiss_retriever.invoke(
    "my favorite fruit is apple"
)

print("=== BM25 ===")
for doc in bm25_docs:
    print(doc.page_content)

print("\n=== FAISS ===")
for doc in faiss_docs:
    print(doc.page_content)

=== BM25 ===
Apple is my favorite company

=== FAISS ===
I like apples


- 검색 시 `config` 매개변수를 통해 검색 설정을 지정합니다.
  - `ensemble_weights` 옵션의 가중치를 [1, 0]으로 설정하여 **모든 검색 결과의 가중치가 BM25 retriever 에 더 많이 부여** 되도록 합니다.

In [14]:
config = {"configureable": {"ensamble_weight": [1, 0]}}   # [1, 0]과 [0, 1]의 의미는 retrievers에 지정한 순서와 연결

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs

[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='5ac3e662-df7f-4f85-b4b7-64c931eddf9f', metadata={}, page_content='I like apples')]

이번에는 검색시 모든 검색 결과의 가중치가 **FAISS retriever 에 더 많이 부여** 되도록 합니다.

In [12]:
config = {"configureable": {"ensamble_weight": [0, 1]}}

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs

[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='5ac3e662-df7f-4f85-b4b7-64c931eddf9f', metadata={}, page_content='I like apples')]

In [16]:
config = {
    "configurable": {
        "ensemble_weights": [1, 0]
    }
}

docs = ensemble_retriever.invoke(
    "my favorite fruit is apple",
    config=config
)

print("=== [1, 0] ===")
for doc in docs:
    print(doc.page_content)


config = {
    "configurable": {
        "ensemble_weights": [0, 1]
    }
}

docs = ensemble_retriever.invoke(
    "my favorite fruit is apple",
    config=config
)

print("\n=== [0, 1] ===")
for doc in docs:
    print(doc.page_content)

=== [1, 0] ===
Apple is my favorite company
I like apples

=== [0, 1] ===
I like apples
Apple is my favorite company
